# Edge-DP KL-Perturbed Dictionary — Bare-Bones Driver

Minimal end-to-end driver for the edge-DP synthetic-graph training approach on **Cora**.

Pipeline:
1. Load Cora; stratified public/private/val splits with inductive edge isolation.
2. Train a **public GCN encoder** on public nodes only → `x_enc` (used as feature space everywhere downstream).
3. Build the **best-performing public dictionary** (`full` mode from `dict_pruning.ipynb`):
   - Stage A: overcomplete candidate pool (anchor-centric prototype key — anchor's 1-hop neighbor mean).
   - Stage B: hard structural / purity / overlap filters.
   - Stage C: k-center diversification + held-out public-query coverage.
4. **Edge-DP exponential mechanism** assignment with **soft-regularized query normalization** (Strategy 5):
$$T_\tau(q)=\frac{q}{\sqrt{\lVert q\rVert_2^2+\tau}}.$$ Globally Lipschitz, so the neg-L2 sensitivity argument for the one-hop mean stays clean.
5. Train a small GCN on the assigned prototype subgraphs and report basic metrics.

In [1]:
# ===================================================================
# CELL 1: Imports & global config
# ===================================================================
import os, time, math, random
from collections import Counter, defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import (
    subgraph, k_hop_subgraph, to_undirected, coalesce, dropout_edge,
)
from torch_geometric.nn import GCNConv, global_mean_pool

device = torch.device('cuda' if torch.cuda.is_available() else
                      'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Active device: {device}')

CONFIG = {
    'seed': 42,
    # splits
    'public_frac': 0.20,
    'val_frac': 0.20,
    'public_query_frac': 0.25,   # within public, held out for Stage-C coverage scoring
    # mechanism
    'epsilon': 1.0,
    'utility_sensitivity': 1.0,
    'query_mode': 'one_hop',
    'walk_hops': 1,
    'query_hops': 1,
    'label_conditioning': True,
    'tau_soft_norm': 1e-3,       # Strategy 5 regularizer (soft denominator clip)
    # dictionary
    'dict_per_class': 8,
    'pool_mult': 6,              # Stage A overcomplete factor
    'min_class_fraction': 1.0,
    'max_proto_nodes': 32,
    # Stage B filters
    'min_proto_nodes': 2,
    'min_proto_edges': 1,
    'purity_floor': 0.5,
    'require_connected': True,
    'max_overlap_frac': 0.85,
    # Stage C
    'kcenter_lambda': 0.5,
    # public encoder
    'encoder_hidden': 64,
    'encoder_out': 32,
    'encoder_epochs': 200,
    'encoder_lr': 0.01,
    'encoder_wd': 5e-4,
    'encoder_dropout': 0.5,
    # downstream GCN training
    'epochs': 50,
    'batch_size': 32,
    'lr': 0.01,
    'feature_jitter_std': 0.05,
    'edge_dropout_p': 0.10,
}

def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG['seed'])

/Users/joyxu/miniforge3/envs/mm-edgeDP/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Active device: mps


In [2]:
# ===================================================================
# CELL 2: Load Cora & build splits with inductive edge isolation
# ===================================================================
print('Loading Cora...')
dataset = Planetoid(root='/tmp/Cora', name='Cora')
data = dataset[0]
data.x = F.normalize(data.x.float(), p=2, dim=1)
data.y = data.y.long()
num_classes = int(dataset.num_classes)
data.edge_index = coalesce(
    to_undirected(data.edge_index, num_nodes=data.num_nodes),
    num_nodes=data.num_nodes,
)
labels = data.y
print(f'Cora: {data.num_nodes} nodes | {data.edge_index.size(1)} edges | {num_classes} classes')


def stratified_split_indices(y, public_frac, val_frac, seed):
    g = torch.Generator().manual_seed(seed)
    pub, val, tr = [], [], []
    for c in torch.unique(y).tolist():
        idx = torch.where(y == c)[0]
        if idx.numel() == 0:
            continue
        perm = idx[torch.randperm(idx.numel(), generator=g)]
        n_pub = max(1, int(public_frac * idx.numel()))
        n_val = max(1, int(val_frac * idx.numel()))
        pub.append(perm[:n_pub])
        val.append(perm[n_pub:n_pub + n_val])
        tr.append(perm[n_pub + n_val:])
    return torch.cat(pub), torch.cat(tr), torch.cat(val)


def split_public_pool_query(public_nodes, labels, query_frac, seed):
    g = torch.Generator().manual_seed(seed)
    pool, query = [], []
    for c in torch.unique(labels[public_nodes]).tolist():
        idx = public_nodes[labels[public_nodes] == c]
        if idx.numel() == 0:
            continue
        perm = idx[torch.randperm(idx.numel(), generator=g)]
        n_q = max(1, int(query_frac * idx.numel()))
        n_q = min(n_q, max(idx.numel() - 1, 0))
        query.append(perm[:n_q])
        pool.append(perm[n_q:])
    return torch.cat(pool), torch.cat(query)


public_nodes, train_nodes, val_nodes = stratified_split_indices(
    labels, CONFIG['public_frac'], CONFIG['val_frac'], seed=CONFIG['seed'])
public_pool_nodes, public_query_nodes = split_public_pool_query(
    public_nodes, labels, CONFIG['public_query_frac'], seed=CONFIG['seed'])

pub_edge_index, _      = subgraph(public_nodes,      data.edge_index, relabel_nodes=False, num_nodes=data.num_nodes)
priv_edge_index, _     = subgraph(train_nodes,       data.edge_index, relabel_nodes=False, num_nodes=data.num_nodes)
val_edge_index, _      = subgraph(val_nodes,         data.edge_index, relabel_nodes=False, num_nodes=data.num_nodes)
pool_edge_index, _     = subgraph(public_pool_nodes, data.edge_index, relabel_nodes=False, num_nodes=data.num_nodes)

print(f'public: {public_nodes.numel()} (pool={public_pool_nodes.numel()}, query={public_query_nodes.numel()}) | '
      f'private train: {train_nodes.numel()} | val: {val_nodes.numel()}')

Loading Cora...


Processing...
/Users/joyxu/miniforge3/envs/mm-edgeDP/lib/python3.11/site-packages/torch_geometric/io/planetoid.py:107: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  out = pickle.load(f, encoding='latin1')
/Users/joyxu/miniforge3/envs/mm-edgeDP/lib/python3.11/site-packages/torch_geometric/io/planetoid.py:107: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  out = pickle.load(f, encoding='latin1')
/Users/joyxu/miniforge3/envs/mm-edgeDP/lib/python3.11/site-packages/torch_geometric/io/planetoid.py:107: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  out = pickle.load(f, encoding='la

Cora: 2708 nodes | 10556 edges | 7 classes
public: 539 (pool=408, query=131) | private train: 1630 | val: 539


Done!


In [3]:
# ===================================================================
# CELL 3: Public GCN encoder (no DP cost) → x_enc
# Trains a 2-layer GCN node classifier on public nodes/edges only.
# We then apply it to all nodes to extract a 32-d penultimate embedding.
# Same-class nodes cluster more tightly in this space, so L2 distance is a
# more meaningful similarity proxy for both Stage-C selection and the EM.
# ===================================================================
class PublicNodeEncoder(nn.Module):
    def __init__(self, in_channels, hidden, out, num_classes, dropout):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden)
        self.conv2 = GCNConv(hidden, out)
        self.head = nn.Linear(out, num_classes)
        self.dropout = dropout

    def encode(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index).relu()
        return x

    def forward(self, x, edge_index):
        return self.head(self.encode(x, edge_index))


def train_public_encoder(x_features, pub_edges, labels, public_nodes, num_classes,
                          hidden, out, epochs, lr, wd, dropout, seed):
    set_seed(seed)
    x_d   = x_features.to(device)
    e_d   = pub_edges.to(device)
    y_d   = labels.to(device)
    pub_mask = torch.zeros(x_d.size(0), dtype=torch.bool, device=device)
    pub_mask[public_nodes] = True

    model = PublicNodeEncoder(x_d.size(1), hidden, out, num_classes, dropout).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    best_acc, best_state = 0.0, None
    for epoch in range(1, epochs + 1):
        model.train(); opt.zero_grad()
        loss = F.cross_entropy(model(x_d, e_d)[pub_mask], y_d[pub_mask])
        loss.backward(); opt.step()
        if epoch % 50 == 0 or epoch == epochs:
            model.eval()
            with torch.no_grad():
                pred = model(x_d, e_d)[pub_mask].argmax(dim=1)
                acc = float((pred == y_d[pub_mask]).float().mean().item())
            if acc > best_acc:
                best_acc = acc
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            print(f'  encoder epoch {epoch:3d} | loss={loss.item():.4f} | pub_acc={acc:.4f}')

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        x_enc = F.normalize(model.encode(x_d, e_d), p=2, dim=1).cpu()
    print(f'Encoder done. x_enc dim={x_enc.size(1)} | best public-node acc={best_acc:.4f}')
    return x_enc


x_enc = train_public_encoder(
    data.x, pub_edge_index, labels, public_nodes, num_classes,
    hidden=CONFIG['encoder_hidden'], out=CONFIG['encoder_out'],
    epochs=CONFIG['encoder_epochs'], lr=CONFIG['encoder_lr'],
    wd=CONFIG['encoder_wd'], dropout=CONFIG['encoder_dropout'],
    seed=CONFIG['seed'],
)

  encoder epoch  50 | loss=0.2541 | pub_acc=0.9722
  encoder epoch 100 | loss=0.1955 | pub_acc=0.9852
  encoder epoch 150 | loss=0.1425 | pub_acc=0.9870
  encoder epoch 200 | loss=0.1712 | pub_acc=0.9852
Encoder done. x_enc dim=32 | best public-node acc=0.9870


In [5]:
# ===================================================================
# CELL 4: Mechanism helpers
#   - one_hop_mean_aggregate
#   - soft_normalize  (Strategy 5: T_tau(q) = q / sqrt(||q||^2 + tau))
#   - build_prototype_feature  (anchor-centric: anchor's 1-hop neighbor mean)
#   - build_private_query_features  (uses soft_normalize, NOT F.normalize)
# ===================================================================
TAU = CONFIG['tau_soft_norm']


def one_hop_mean_aggregate(edge_index, x_features, num_nodes):
    """H[u] = mean_{v in N(u)} x_v on the given (sub)graph edge set."""
    eu = coalesce(to_undirected(edge_index, num_nodes=num_nodes), num_nodes=num_nodes)
    H = torch.zeros_like(x_features)
    if eu.numel() == 0:
        return H
    row, col = eu
    H.scatter_add_(0, row.unsqueeze(1).expand(-1, x_features.size(1)), x_features[col])
    counts = torch.zeros((num_nodes,), dtype=x_features.dtype, device=x_features.device)
    counts.scatter_add_(0, row, torch.ones_like(row, dtype=x_features.dtype))
    out = torch.zeros_like(H)
    nz = counts > 0
    out[nz] = H[nz] / counts[nz].unsqueeze(1)
    return out


def soft_normalize(x, tau=TAU, dim=-1):
    """Soft regularized normalization (Strategy 5):
        T_tau(q) = q / sqrt(||q||^2 + tau)
    Globally bounded (||T_tau(q)|| < 1) and globally Lipschitz, so the neg-L2
    utility on the one-hop mean has bounded sensitivity even near zero.
    Approaches q / ||q||_2 when ||q||_2 >> sqrt(tau), preserving the angular
    matching geometry against L2-normalized prototypes.
    """
    sq = (x * x).sum(dim=dim, keepdim=True)
    return x / torch.sqrt(sq + tau)


def build_prototype_feature(edge_index_sub, x_sub, anchor_local_idx, query_mode='one_hop'):
    """Anchor-centric prototype key (mirrors the per-node private query):
        proto_feat = normalize(H1_nodes[anchor_local_idx])
    where H1_nodes is the 1-hop neighbor mean computed on the prototype subgraph.
    """
    n = x_sub.size(0)
    H1_nodes = one_hop_mean_aggregate(edge_index_sub, x_sub, num_nodes=n)
    z1 = H1_nodes[anchor_local_idx]
    if query_mode == 'one_hop':
        return F.normalize(z1, p=2, dim=0)
    if query_mode == 'two_hop_concat':
        H2_nodes = one_hop_mean_aggregate(edge_index_sub, H1_nodes, num_nodes=n)
        z2 = H2_nodes[anchor_local_idx]
        return F.normalize(torch.cat([z1, z2], dim=0), p=2, dim=0)
    raise ValueError(f'Unsupported query_mode: {query_mode}')


def build_private_query_features(private_edges, x_features, query_mode='one_hop', tau=TAU):
    """Per-private-node query vector with soft regularized normalization.
    Replaces the previous F.normalize(...) (which is not globally Lipschitz at zero).
    """
    n = x_features.size(0)
    H1 = one_hop_mean_aggregate(private_edges, x_features, num_nodes=n)
    if query_mode == 'one_hop':
        return soft_normalize(H1, tau=tau, dim=1)
    if query_mode == 'two_hop_concat':
        H2 = one_hop_mean_aggregate(private_edges, H1, num_nodes=n)
        return soft_normalize(torch.cat([H1, H2], dim=1), tau=tau, dim=1)
    raise ValueError(f'Unsupported query_mode: {query_mode}')


def _is_connected_subgraph(node_ids, edge_index, num_nodes):
    if node_ids.numel() <= 1:
        return True
    sub_edges, _ = subgraph(node_ids, edge_index, relabel_nodes=True, num_nodes=num_nodes)
    if sub_edges.numel() == 0:
        return False
    n = node_ids.numel()
    adj = defaultdict(list)
    for a, b in zip(sub_edges[0].tolist(), sub_edges[1].tolist()):
        adj[a].append(b)
    seen = {0}; stack = [0]
    while stack:
        u = stack.pop()
        for v in adj[u]:
            if v not in seen:
                seen.add(v); stack.append(v)
    return len(seen) == n

In [6]:
# ===================================================================
# CELL 5: Build the public dictionary (full pipeline = best mode)
#   Stage A — overcomplete candidate pool (anchor-centric proto_feat).
#   Stage B — hard structural / purity / overlap filters.
#   Stage C — k-center diversification + held-out public-query coverage.
# All stages use x_enc. No private edges are touched here.
# ===================================================================

def build_candidate_pool(data_obj, x_features, labels, pool_nodes, pool_edges,
                          num_classes, target_per_class, num_hops, query_mode,
                          max_proto_nodes, min_class_fraction, rng_seed):
    pool_und = coalesce(
        to_undirected(pool_edges, num_nodes=data_obj.num_nodes),
        num_nodes=data_obj.num_nodes,
    )
    g = torch.Generator().manual_seed(int(rng_seed))
    pool_labels = labels[pool_nodes]
    candidates = []
    for c in range(num_classes):
        class_pool = pool_nodes[pool_labels == c]
        if class_pool.numel() == 0:
            continue
        for _ in range(target_per_class):
            pick = torch.randint(0, class_pool.numel(), (1,), generator=g).item()
            anchor = class_pool[pick]
            subset, _, _, _ = k_hop_subgraph(
                int(anchor.item()), num_hops=num_hops,
                edge_index=pool_und, relabel_nodes=False,
                num_nodes=data_obj.num_nodes,
            )
            proto_labels = labels[subset]
            class_mask = proto_labels == c
            class_count = int(class_mask.sum().item())
            subset_count = int(subset.numel())
            frac = class_count / max(subset_count, 1)

            if frac >= min_class_fraction and class_count > 0:
                class_subset = subset
            else:
                class_subset = subset[class_mask]
                if class_subset.numel() == 0:
                    class_subset = anchor.view(1)

            # Anchor must always be in the prototype subgraph (anchor-centric proto_feat).
            if not torch.any(class_subset == anchor):
                class_subset = torch.unique(torch.cat([class_subset, anchor.view(1)]))

            if class_subset.numel() > max_proto_nodes:
                non_anchor = class_subset[class_subset != anchor]
                budget = max_proto_nodes - 1
                if budget <= 0:
                    class_subset = anchor.view(1)
                else:
                    if non_anchor.numel() > budget:
                        perm = torch.randperm(non_anchor.numel(), generator=g)[:budget]
                        non_anchor = non_anchor[perm]
                    class_subset = torch.cat([anchor.view(1), non_anchor])

            sub_edges, _ = subgraph(class_subset, pool_und, relabel_nodes=True,
                                     num_nodes=data_obj.num_nodes)
            x_sub = x_features[class_subset]
            anchor_local_idx = int((class_subset == anchor).nonzero(as_tuple=True)[0].item())
            proto_feat = build_prototype_feature(sub_edges, x_sub, anchor_local_idx, query_mode)

            candidates.append({
                'class': c,
                'anchor': int(anchor.item()),
                'node_ids': class_subset.clone(),
                'x': x_sub,
                'edge_index': sub_edges,
                'proto_feat': proto_feat,
                'n_nodes': int(class_subset.numel()),
                'n_edges': int(sub_edges.size(1)),
                'purity': frac,
            })
    return candidates


def hard_filter_pool(pool, pub_edges, num_nodes, min_nodes, min_edges, purity_floor,
                      require_connected, max_overlap_frac):
    pub_und = coalesce(to_undirected(pub_edges, num_nodes=num_nodes), num_nodes=num_nodes)
    kept_per_class = defaultdict(list); kept = []; reasons = Counter()
    for p in pool:
        if p['n_nodes'] < min_nodes:    reasons['min_nodes'] += 1; continue
        if p['n_edges'] < min_edges:    reasons['min_edges'] += 1; continue
        if p['purity']  < purity_floor: reasons['purity']    += 1; continue
        if require_connected and not _is_connected_subgraph(p['node_ids'], pub_und, num_nodes):
            reasons['disconnected'] += 1; continue
        s = set(p['node_ids'].tolist()); overlap = False
        for prev_set, _ in kept_per_class[p['class']]:
            inter = len(s & prev_set)
            if inter / max(len(s | prev_set), 1) >= max_overlap_frac:
                overlap = True; break
        if overlap:
            reasons['overlap'] += 1; continue
        kept_per_class[p['class']].append((s, p))
        kept.append(p)
    return kept, reasons


@torch.no_grad()
def compute_public_query_features(query_nodes, pool_edges, x_features, labels, num_classes,
                                    query_mode, query_hops, num_total_nodes):
    """Per held-out public-query-node feature in the same (anchor-centric) space as proto_feat."""
    pool_und = coalesce(
        to_undirected(pool_edges, num_nodes=num_total_nodes),
        num_nodes=num_total_nodes,
    )
    per_class = {c: [] for c in range(num_classes)}
    for node in query_nodes.tolist():
        subset, _, _, _ = k_hop_subgraph(
            node, num_hops=query_hops, edge_index=pool_und,
            relabel_nodes=False, num_nodes=num_total_nodes,
        )
        if subset.numel() == 0:
            subset = torch.tensor([node], dtype=torch.long)
        if not torch.any(subset == node):
            subset = torch.unique(torch.cat([subset, torch.tensor([node])]))
        sub_edges, _ = subgraph(subset, pool_und, relabel_nodes=True, num_nodes=num_total_nodes)
        anchor_local_idx = int((subset == node).nonzero(as_tuple=True)[0].item())
        q_feat = build_prototype_feature(sub_edges, x_features[subset], anchor_local_idx, query_mode)
        per_class[int(labels[node].item())].append(q_feat)
    return {c: torch.stack(L, dim=0) if L else torch.empty(0) for c, L in per_class.items()}


@torch.no_grad()
def diversify_and_cover(kept_pool, candidate_pool, query_features_per_class,
                         num_classes, dict_per_class, kcenter_lambda, rng_seed):
    g = torch.Generator().manual_seed(int(rng_seed))
    by_kept = defaultdict(list)
    for p in kept_pool: by_kept[p['class']].append(p)
    by_all = defaultdict(list)
    for p in candidate_pool: by_all[p['class']].append(p)

    selected = []
    class_to_proto = {c: [] for c in range(num_classes)}
    for c in range(num_classes):
        cands = list(by_kept.get(c, []))
        if len(cands) < dict_per_class:
            cand_ids = {id(p) for p in cands}
            spares = [p for p in by_all.get(c, []) if id(p) not in cand_ids]
            random.Random(int(rng_seed) + c).shuffle(spares)
            cands += spares[:max(dict_per_class - len(cands), 0)]
        if len(cands) == 0:
            continue
        feats = torch.stack([p['proto_feat'] for p in cands], dim=0)
        Q = query_features_per_class.get(c, None)
        if Q is None or Q.numel() == 0:
            start_idx = int(torch.randint(0, len(cands), (1,), generator=g).item())
            running_cov = None
        else:
            single_cov = torch.cdist(Q, feats, p=2).mean(dim=0)
            start_idx = int(single_cov.argmin().item())
            running_cov = torch.linalg.norm(Q - feats[start_idx].unsqueeze(0), dim=1)
        chosen = [start_idx]
        dist_to_sel = torch.linalg.norm(feats - feats[start_idx].unsqueeze(0), dim=1)
        budget = min(dict_per_class, len(cands))
        while len(chosen) < budget:
            div_score = dist_to_sel
            if running_cov is not None:
                d_to_cands = torch.cdist(Q, feats, p=2)
                new_cov = torch.minimum(running_cov.unsqueeze(1), d_to_cands).mean(dim=0)
                cov_score = -new_cov
            else:
                cov_score = torch.zeros(len(cands))
            score = kcenter_lambda * div_score + (1.0 - kcenter_lambda) * cov_score
            for ci in chosen: score[ci] = float('-inf')
            nxt = int(score.argmax().item())
            chosen.append(nxt)
            dist_to_sel = torch.minimum(
                dist_to_sel,
                torch.linalg.norm(feats - feats[nxt].unsqueeze(0), dim=1),
            )
            if running_cov is not None:
                running_cov = torch.minimum(
                    running_cov,
                    torch.linalg.norm(Q - feats[nxt].unsqueeze(0), dim=1),
                )
        for li in chosen:
            class_to_proto[c].append(len(selected))
            selected.append(cands[li])
    return selected, class_to_proto


set_seed(CONFIG['seed'])
print('Stage A: building overcomplete candidate pool...')
candidate_pool = build_candidate_pool(
    data_obj=data, x_features=x_enc, labels=labels,
    pool_nodes=public_pool_nodes, pool_edges=pool_edge_index,
    num_classes=num_classes,
    target_per_class=CONFIG['pool_mult'] * CONFIG['dict_per_class'],
    num_hops=CONFIG['walk_hops'], query_mode=CONFIG['query_mode'],
    max_proto_nodes=CONFIG['max_proto_nodes'],
    min_class_fraction=CONFIG['min_class_fraction'],
    rng_seed=CONFIG['seed'],
)
print(f'  pool size = {len(candidate_pool)}')

print('Stage B: hard filter...')
kept_pool, drop_reasons = hard_filter_pool(
    candidate_pool, pub_edge_index, data.num_nodes,
    min_nodes=CONFIG['min_proto_nodes'], min_edges=CONFIG['min_proto_edges'],
    purity_floor=CONFIG['purity_floor'],
    require_connected=CONFIG['require_connected'],
    max_overlap_frac=CONFIG['max_overlap_frac'],
)
print(f'  kept = {len(kept_pool)} | drops = {dict(drop_reasons)}')

print('Stage C: diversify + cover (k-center on x_enc)...')
query_features_per_class = compute_public_query_features(
    public_query_nodes, pool_edge_index, x_enc, labels, num_classes,
    CONFIG['query_mode'], CONFIG['query_hops'], data.num_nodes,
)
public_dict, class_to_proto_indices = diversify_and_cover(
    kept_pool, candidate_pool, query_features_per_class,
    num_classes=num_classes,
    dict_per_class=CONFIG['dict_per_class'],
    kcenter_lambda=CONFIG['kcenter_lambda'],
    rng_seed=CONFIG['seed'],
)
public_proto_feats = torch.stack([p['proto_feat'] for p in public_dict], dim=0)
print(f'  dictionary size = {len(public_dict)} | feature dim = {public_proto_feats.size(1)}')
print(f'  prototypes per class: '
      f'{ {c: len(idxs) for c, idxs in class_to_proto_indices.items()} }')

Stage A: building overcomplete candidate pool...
  pool size = 336
Stage B: hard filter...
  kept = 50 | drops = {'min_nodes': 245, 'overlap': 40, 'purity': 1}
Stage C: diversify + cover (k-center on x_enc)...
  dictionary size = 56 | feature dim = 32
  prototypes per class: {0: 8, 1: 8, 2: 8, 3: 8, 4: 8, 5: 8, 6: 8}


In [8]:
# ===================================================================
# CELL 6: Edge-DP exponential mechanism (per-private-node prototype assignment)
#   - Private query: q_u = soft_normalize(H1[u]) on private edges (Strategy 5).
#   - Utility: u(q, k) = -||proto_feat_k - q||_2  (neg-L2).
#   - Edge-level composition: a single private edge affects <=2 node queries,
#     so we split the total epsilon over two releases.
# ===================================================================

def gumbel_max_sample(logits):
    u = torch.rand_like(logits).clamp_(1e-8, 1 - 1e-8)
    return torch.argmax(logits + (-torch.log(-torch.log(u)))).item()


@torch.no_grad()
def synthesize_edge_dp_assignments(private_edges, x_features, labels, private_nodes,
                                     dict_features, class_to_proto_indices,
                                     epsilon_total, utility_sensitivity,
                                     query_mode, label_conditioning, tau):
    eps_per = epsilon_total / 2.0
    Q = build_private_query_features(private_edges, x_features, query_mode=query_mode, tau=tau)
    K = dict_features.size(0)
    nc = int(labels.max().item()) + 1
    all_idx = torch.arange(K, dtype=torch.long)
    class_idx = {
        c: (torch.tensor(class_to_proto_indices.get(c, []), dtype=torch.long)
            if class_to_proto_indices.get(c) else all_idx)
        for c in range(nc)
    }
    N = int(private_nodes.numel())
    sel = torch.empty(N, dtype=torch.long)
    priv_labels = labels[private_nodes].long().clone()
    counts = torch.zeros(K, dtype=torch.long)
    ent_sum = top_sum = 0.0
    for j, u in enumerate(private_nodes.tolist()):
        y_u = int(labels[u].item())
        cand_idx = class_idx[y_u] if label_conditioning else all_idx
        cand_feats = dict_features.index_select(0, cand_idx)
        d = torch.linalg.norm(cand_feats - Q[u].unsqueeze(0), dim=1)
        logits = (eps_per / (2.0 * utility_sensitivity)) * (-d)
        probs = torch.softmax(logits, dim=0)
        ent_sum += float(-(probs * torch.log(probs.clamp_min(1e-12))).sum().item())
        top_sum += float(probs.max().item())
        li = gumbel_max_sample(logits)
        pi = int(cand_idx[li].item())
        sel[j] = pi
        counts[pi] += 1
    diag = {
        'epsilon_total': float(epsilon_total),
        'epsilon_per_node': float(eps_per),
        'mean_entropy': ent_sum / max(N, 1),
        'mean_top_probability': top_sum / max(N, 1),
        'unique_selected_ratio': float((counts > 0).sum().item()) / float(max(K, 1)),
    }
    return {'proto_indices': sel, 'labels': priv_labels}, diag


set_seed(CONFIG['seed'])
print(f'Edge-DP exponential mechanism (epsilon_total={CONFIG["epsilon"]:.3f}, tau={TAU})')
assignments, syn_diag = synthesize_edge_dp_assignments(
    priv_edge_index, x_enc, labels, train_nodes,
    public_proto_feats, class_to_proto_indices,
    epsilon_total=CONFIG['epsilon'],
    utility_sensitivity=CONFIG['utility_sensitivity'],
    query_mode=CONFIG['query_mode'],
    label_conditioning=CONFIG['label_conditioning'],
    tau=CONFIG['tau_soft_norm'],
)
print(f'  epsilon_per_node      = {syn_diag["epsilon_per_node"]:.4f}')
print(f'  mean entropy          = {syn_diag["mean_entropy"]:.4f}')
print(f'  mean top-1 prob       = {syn_diag["mean_top_probability"]:.4f}')
print(f'  unique-prototype-ratio= {syn_diag["unique_selected_ratio"]:.2%}')

Edge-DP exponential mechanism (epsilon_total=1.000, tau=0.001)
  epsilon_per_node      = 0.5000
  mean entropy          = 2.0788
  mean top-1 prob       = 0.1288
  unique-prototype-ratio= 100.00%


In [9]:
# ===================================================================
# CELL 7: Train a small GCN on the assigned prototype subgraphs
# (post-processing of DP-safe assignments, no extra privacy cost).
# ===================================================================
class PrototypeAssignmentDataset(torch.utils.data.Dataset):
    def __init__(self, public_dict, proto_indices, labels):
        self.public_dict = public_dict
        self.proto_indices = proto_indices.long().cpu()
        self.labels = labels.long().cpu()
    def __len__(self):
        return self.proto_indices.numel()
    def __getitem__(self, idx):
        p = self.public_dict[int(self.proto_indices[idx].item())]
        return Data(x=p['x'], edge_index=p['edge_index'], y=self.labels[idx].view(1))


class StandardGCN(nn.Module):
    def __init__(self, hidden, num_features, num_classes):
        super().__init__()
        self.conv1 = GCNConv(num_features, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.lin = nn.Linear(hidden, num_classes)
    def forward(self, x, edge_index, batch):
        x = self.conv1(x, edge_index).relu()
        x = self.conv2(x, edge_index).relu()
        x = global_mean_pool(x, batch)
        return self.lin(x)


def build_validation_loader(x_features, labels, val_nodes, val_edges, query_hops, num_total_nodes):
    vu = coalesce(to_undirected(val_edges, num_nodes=num_total_nodes), num_nodes=num_total_nodes)
    dl = []
    for n in val_nodes.tolist():
        subset, sei, _, _ = k_hop_subgraph(
            n, num_hops=query_hops, edge_index=vu,
            relabel_nodes=True, num_nodes=num_total_nodes,
        )
        dl.append(Data(x=x_features[subset], edge_index=sei, y=labels[n].unsqueeze(0)))
    return DataLoader(dl, batch_size=32, shuffle=False)


def train_gnn(assignments, public_dict, val_loader, num_features, num_classes,
                epochs, batch_size, lr, feature_jitter_std, edge_dropout_p):
    ds = PrototypeAssignmentDataset(public_dict, assignments['proto_indices'], assignments['labels'])
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True)
    model = StandardGCN(32, num_features, num_classes).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()
    best = 0.0
    for epoch in range(1, epochs + 1):
        model.train(); total_loss = 0.0; train_correct = 0
        for batch in dl:
            batch = batch.to(device)
            opt.zero_grad(set_to_none=True)
            xa = batch.x + torch.randn_like(batch.x) * feature_jitter_std
            ea, _ = dropout_edge(batch.edge_index, p=edge_dropout_p)
            if ea.numel() == 0:
                ea = batch.edge_index
            out = model(xa, ea, batch.batch)
            loss = crit(out, batch.y)
            loss.backward(); opt.step()
            total_loss += loss.item() * batch.num_graphs
            train_correct += int((out.argmax(dim=1) == batch.y).sum())
        train_acc = train_correct / len(dl.dataset)

        model.eval(); val_correct = 0
        with torch.no_grad():
            for vb in val_loader:
                vb = vb.to(device)
                pred = model(vb.x, vb.edge_index, vb.batch).argmax(dim=1)
                val_correct += int((pred == vb.y).sum())
        val_acc = val_correct / len(val_loader.dataset)
        best = max(best, val_acc)
        if epoch % 10 == 0 or epoch == 1:
            print(f'  epoch {epoch:03d} | train_loss={total_loss/len(dl.dataset):.4f} | '
                  f'train_acc={train_acc:.4f} | val_acc={val_acc:.4f}')
    return best


val_loader = build_validation_loader(
    x_enc, labels, val_nodes, val_edge_index, CONFIG['query_hops'], data.num_nodes,
)
print(f'Validation set: {len(val_loader.dataset)} graphs')

# Majority-class baseline
val_counts = torch.bincount(labels[val_nodes], minlength=num_classes)
majority_acc = float((labels[val_nodes] == int(val_counts.argmax().item())).float().mean().item())
print(f'Majority-class baseline (val): {majority_acc:.4f}')

print('\n--- Training GCN on edge-DP synthetic assignments ---')
set_seed(CONFIG['seed'])
best_val = train_gnn(
    assignments, public_dict, val_loader,
    num_features=x_enc.size(1), num_classes=num_classes,
    epochs=CONFIG['epochs'], batch_size=CONFIG['batch_size'], lr=CONFIG['lr'],
    feature_jitter_std=CONFIG['feature_jitter_std'],
    edge_dropout_p=CONFIG['edge_dropout_p'],
)

print('\n=== Summary ===')
print(f'Dataset                : Cora')
print(f'Epsilon (total)        : {CONFIG["epsilon"]}')
print(f'Tau (soft-norm)        : {CONFIG["tau_soft_norm"]}')
print(f'Dictionary size        : {len(public_dict)}')
print(f'Mean selection entropy : {syn_diag["mean_entropy"]:.4f}')
print(f'Mean top-1 probability : {syn_diag["mean_top_probability"]:.4f}')
print(f'Majority baseline acc  : {majority_acc:.4f}')
print(f'Best validation acc    : {best_val:.4f}')

Validation set: 539 graphs
Majority-class baseline (val): 0.3024

--- Training GCN on edge-DP synthetic assignments ---
  epoch 001 | train_loss=0.6273 | train_acc=0.8098 | val_acc=0.6122
  epoch 010 | train_loss=0.0004 | train_acc=1.0000 | val_acc=0.6122
  epoch 020 | train_loss=0.0001 | train_acc=1.0000 | val_acc=0.6234
  epoch 030 | train_loss=0.0000 | train_acc=1.0000 | val_acc=0.6160
  epoch 040 | train_loss=0.0000 | train_acc=1.0000 | val_acc=0.6215
  epoch 050 | train_loss=0.0000 | train_acc=1.0000 | val_acc=0.6308

=== Summary ===
Dataset                : Cora
Epsilon (total)        : 1.0
Tau (soft-norm)        : 0.001
Dictionary size        : 56
Mean selection entropy : 2.0788
Mean top-1 probability : 0.1288
Majority baseline acc  : 0.3024
Best validation acc    : 0.6327


In [10]:
# ===================================================================
# CELL 8: Baseline 1 — Gaussian SGC
# Mean-aggregate 1-hop neighbors, add calibrated Gaussian noise,
# train MLP on the noisy node embeddings.
# ===================================================================
import math

def calibrate_gaussian_sigma(epsilon, delta, sensitivity=1.0):
    """Standard Gaussian mechanism: σ = sqrt(2 ln(1.25/δ)) · Δ / ε"""
    return math.sqrt(2.0 * math.log(1.25 / delta)) * sensitivity / epsilon


class NodeMLP(nn.Module):
    def __init__(self, in_dim, hidden, num_classes, dropout=0.5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, num_classes),
        )
    def forward(self, x):
        return self.net(x)


def train_node_mlp(x_train, y_train, x_val, y_val,
                   in_dim, hidden, num_classes,
                   epochs=300, lr=0.01, wd=5e-4, dropout=0.5, seed=42):
    set_seed(seed)
    model = NodeMLP(in_dim, hidden, num_classes, dropout).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    xtr = x_train.to(device); ytr = y_train.to(device)
    xvl = x_val.to(device);   yvl = y_val.to(device)
    best = 0.0
    for ep in range(1, epochs + 1):
        model.train(); opt.zero_grad()
        F.cross_entropy(model(xtr), ytr).backward(); opt.step()
        if ep % 100 == 0 or ep == epochs:
            model.eval()
            with torch.no_grad():
                acc = float((model(xvl).argmax(1) == yvl).float().mean())
            best = max(best, acc)
            print(f'  epoch {ep:3d} | val_acc={acc:.4f}')
    return best


DELTA = 1e-5
sigma_gauss = calibrate_gaussian_sigma(CONFIG['epsilon'], DELTA, sensitivity=1.0)
print(f'Gaussian SGC  ε={CONFIG["epsilon"]}  δ={DELTA}  σ={sigma_gauss:.4f}')

# Noisy 1-hop mean aggregation on private edges (train nodes)
set_seed(CONFIG['seed'])
H_priv = one_hop_mean_aggregate(priv_edge_index, x_enc, num_nodes=data.num_nodes)
no_nbr = H_priv.norm(dim=1) == 0          # isolated nodes → use self-feature
H_priv[no_nbr] = x_enc[no_nbr]
x_sgc_train = (H_priv[train_nodes]
               + torch.randn(train_nodes.numel(), x_enc.size(1)) * sigma_gauss).detach()

# Same treatment for val nodes (consistent noise at inference)
H_val_agg = one_hop_mean_aggregate(val_edge_index, x_enc, num_nodes=data.num_nodes)
no_nbr_v = H_val_agg.norm(dim=1) == 0
H_val_agg[no_nbr_v] = x_enc[no_nbr_v]
x_sgc_val = (H_val_agg[val_nodes]
             + torch.randn(val_nodes.numel(), x_enc.size(1)) * sigma_gauss).detach()

print('\n--- Training Gaussian SGC MLP ---')
sgc_best_val = train_node_mlp(
    x_sgc_train, labels[train_nodes],
    x_sgc_val,   labels[val_nodes],
    in_dim=x_enc.size(1), hidden=64, num_classes=num_classes,
    epochs=300, lr=0.01, seed=CONFIG['seed'],
)
print(f'\nGaussian SGC best val acc: {sgc_best_val:.4f}')

Gaussian SGC  ε=1.0  δ=1e-05  σ=4.8448

--- Training Gaussian SGC MLP ---
  epoch 100 | val_acc=0.2746
  epoch 200 | val_acc=0.2727
  epoch 300 | val_acc=0.2560

Gaussian SGC best val acc: 0.2746


In [11]:
# ===================================================================
# CELL 9: Baseline 2 — Edge Randomized Response (Edge-RR)
# Keep each existing edge with prob  p = e^ε / (e^ε + 1).
# Add each non-edge        with prob  q = 1  / (e^ε + 1).
# Train a standard 2-layer node-classification GCN on the perturbed graph.
# ===================================================================

def apply_edge_rr(edge_index, node_ids, epsilon, num_nodes, seed):
    """
    Vectorised edge-level randomised response over all pairs among node_ids.
    Returns a perturbed edge_index in global node IDs (undirected).
    """
    torch.manual_seed(seed)
    p_keep = math.exp(epsilon) / (math.exp(epsilon) + 1.0)
    p_add  = 1.0 / (math.exp(epsilon) + 1.0)

    n = node_ids.numel()
    node_list = node_ids.tolist()
    node_set  = set(node_list)
    local     = {v: i for i, v in enumerate(node_list)}

    # Build local upper-triangle adjacency (n x n bool)
    adj = torch.zeros(n, n, dtype=torch.bool)
    for r, c in zip(edge_index[0].tolist(), edge_index[1].tolist()):
        if r in node_set and c in node_set and r < c:
            adj[local[r], local[c]] = True

    ti, tj = torch.triu_indices(n, n, offset=1)
    exist  = adj[ti, tj]

    probs = torch.where(exist,
                        torch.full(exist.shape, p_keep),
                        torch.full(exist.shape, p_add))
    keep  = torch.rand(probs.shape) < probs

    ki = ti[keep]; kj = tj[keep]
    node_arr = node_ids
    src = torch.cat([node_arr[ki], node_arr[kj]])
    dst = torch.cat([node_arr[kj], node_arr[ki]])

    n_orig  = int(exist.sum())
    n_kept  = int(keep[exist].sum())
    n_added = int(keep[~exist].sum())
    print(f'  Edge-RR (ε={epsilon}): {n_orig} real edges → '
          f'kept {n_kept} (dropped {n_orig - n_kept}) + {n_added} fake  '
          f'(flip prob = {1 - p_keep:.3f})')
    return torch.stack([src, dst])


class NodeGCN(nn.Module):
    """2-layer GCN for node classification (inductive)."""
    def __init__(self, in_dim, hidden, num_classes, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.conv2 = GCNConv(hidden, num_classes)
        self.dropout = dropout

    def forward(self, x, edge_index):
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.conv2(x, edge_index)


def train_rr_gcn(perturbed_ei, x_features, train_ids, val_ids, val_ei,
                 labels, num_features, num_classes,
                 hidden=64, epochs=300, lr=0.01, dropout=0.5,
                 num_nodes=None, seed=42):
    set_seed(seed)
    # Relabel train subgraph to contiguous local indices
    tr_ei, _ = subgraph(train_ids, perturbed_ei, relabel_nodes=True, num_nodes=num_nodes)
    tr_ei = coalesce(to_undirected(tr_ei, num_nodes=train_ids.numel()),
                     num_nodes=train_ids.numel())
    xtr   = x_features[train_ids].to(device)
    ytr   = labels[train_ids].to(device)
    tr_d  = tr_ei.to(device)

    # Relabel val subgraph
    vl_ei, _ = subgraph(val_ids, val_ei, relabel_nodes=True, num_nodes=num_nodes)
    vl_ei = coalesce(to_undirected(vl_ei, num_nodes=val_ids.numel()),
                     num_nodes=val_ids.numel())
    xvl  = x_features[val_ids].to(device)
    yvl  = labels[val_ids].to(device)
    vl_d = vl_ei.to(device)

    model = NodeGCN(num_features, hidden, num_classes, dropout).to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    best  = 0.0
    for ep in range(1, epochs + 1):
        model.train(); opt.zero_grad()
        F.cross_entropy(model(xtr, tr_d), ytr).backward(); opt.step()
        if ep % 100 == 0 or ep == epochs:
            model.eval()
            with torch.no_grad():
                acc = float((model(xvl, vl_d).argmax(1) == yvl).float().mean())
            best = max(best, acc)
            print(f'  epoch {ep:3d} | val_acc={acc:.4f}')
    return best


print('=== Edge-RR Baseline ===')
set_seed(CONFIG['seed'])
perturbed_ei = apply_edge_rr(
    priv_edge_index, train_nodes, CONFIG['epsilon'], data.num_nodes, CONFIG['seed'])

print('\n--- Training GCN on perturbed graph ---')
rr_best_val = train_rr_gcn(
    perturbed_ei, x_enc, train_nodes, val_nodes, val_edge_index, labels,
    num_features=x_enc.size(1), num_classes=num_classes,
    hidden=64, epochs=300, lr=0.01, num_nodes=data.num_nodes, seed=CONFIG['seed'],
)
print(f'\nEdge-RR best val acc: {rr_best_val:.4f}')

=== Edge-RR Baseline ===
  Edge-RR (ε=1.0): 1002 real edges → kept 733 (dropped 269) + 357313 fake  (flip prob = 0.269)

--- Training GCN on perturbed graph ---
  epoch 100 | val_acc=0.3024
  epoch 200 | val_acc=0.3024
  epoch 300 | val_acc=0.3024

Edge-RR best val acc: 0.3024


In [12]:
# ===================================================================
# CELL 10: Baseline 3 — GAP-EDP (Sajadmanesh et al., 2022)
# Architecture: Encoder (MLP, no edges) → PMA (noisy sum agg, K hops)
#               → Classifier (MLP on concatenated hop features).
# Edge-DP via Gaussian mechanism on sum-aggregated embeddings.
# Sensitivity Δ = 1 (L2-normalised features, sum agg — GAP Lemma 1).
# ===================================================================

def gap_calibrate_sigma(epsilon, delta, sensitivity=1.0, K=1):
    """
    Bisection solve for σ s.t. K hops of Gaussian mechanism are (ε,δ)-DP.
    Uses Proposition 2 of Sajadmanesh et al.:  ε = K·Δ²/(2σ²) + sqrt(2K·ln(1/δ))·Δ/σ
    """
    c = math.sqrt(2.0 * K * math.log(1.0 / delta)) * sensitivity
    def eps_fn(s):
        return K * sensitivity**2 / (2.0 * s**2) + c / s
    lo, hi = 1e-8, 1e5
    for _ in range(200):
        mid = (lo + hi) / 2.0
        (lo if eps_fn(mid) > epsilon else hi).__class__  # dummy; real assignment below
        if eps_fn(mid) > epsilon:
            lo = mid
        else:
            hi = mid
    return hi


class GAPEncoder(nn.Module):
    """2-layer MLP encoder — no edges used."""
    def __init__(self, in_dim, hidden, out_dim, num_classes, dropout=0.5):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, out_dim), nn.ReLU(),
        )
        self.head = nn.Linear(out_dim, num_classes)

    def encode(self, x): return self.enc(x)
    def forward(self, x): return self.head(self.encode(x))


@torch.no_grad()
def private_multi_hop_agg(edge_index, x_norm, node_ids, sigma, num_nodes, K, seed):
    """
    GAP Private Multi-hop Aggregation (Algorithm 1 in the paper).
    For k = 1..K:  X^(k) = A^T · X̂^(k-1)  (sum agg)
                   X̃^(k) = X^(k) + N(0, σ²I)   (Gaussian noise on target nodes only)
                   X̂^(k) = X̃^(k) / ||X̃^(k)||   (row-normalise)
    Returns list of K perturbed + normalised hop tensors for node_ids.
    """
    torch.manual_seed(seed)
    eu = coalesce(to_undirected(edge_index, num_nodes=num_nodes), num_nodes=num_nodes)
    dst, src = eu   # edge goes src → dst  (A^T · X means sum over sources per destination)
    d = x_norm.size(1)
    x_hat = x_norm.clone()   # X̂^(0)
    hops  = []
    for _ in range(K):
        # Sum aggregation: X^(k)[dst] = Σ_{src connected to dst} X̂^(k-1)[src]
        X_agg = torch.zeros_like(x_hat)
        X_agg.scatter_add_(0, dst.unsqueeze(1).expand(-1, d), x_hat[src])
        # Perturb only the target node rows
        X_tilde = X_agg.clone()
        X_tilde[node_ids] = X_tilde[node_ids] + torch.randn(node_ids.numel(), d) * sigma
        # Row-normalise all rows for the next hop
        norms = X_tilde.norm(dim=1, keepdim=True).clamp(min=1e-8)
        x_hat = X_tilde / norms
        hops.append(x_hat[node_ids].clone())
    return hops


GAP_K       = 1     # number of hops (K=1 is standard for edge-level DP)
GAP_ENC_DIM = 32    # encoder output dim (matches our method's x_enc)

sigma_gap = gap_calibrate_sigma(CONFIG['epsilon'], DELTA, sensitivity=1.0, K=GAP_K)
print(f'GAP-EDP  ε={CONFIG["epsilon"]}  δ={DELTA}  K={GAP_K}  σ={sigma_gap:.4f}')

# ------------------------------------------------------------------
# Step 1: train MLP encoder on private train nodes (no edges)
# ------------------------------------------------------------------
print('\n--- Step 1: GAP encoder (MLP, no edges) ---')
set_seed(CONFIG['seed'])
gap_enc = GAPEncoder(data.x.size(1), hidden=64, out_dim=GAP_ENC_DIM,
                     num_classes=num_classes, dropout=0.5).to(device)
gap_enc_opt = torch.optim.Adam(gap_enc.parameters(), lr=0.01, weight_decay=5e-4)
xraw_tr = data.x[train_nodes].to(device)
yraw_tr = labels[train_nodes].to(device)
for ep in range(1, 201):
    gap_enc.train(); gap_enc_opt.zero_grad()
    F.cross_entropy(gap_enc(xraw_tr), yraw_tr).backward(); gap_enc_opt.step()
gap_enc.eval()
with torch.no_grad():
    tr_acc = float((gap_enc(xraw_tr).argmax(1) == yraw_tr).float().mean())
print(f'  encoder train acc (ep 200): {tr_acc:.4f}')

# Encode all nodes and row-normalise → X̂^(0)
with torch.no_grad():
    x_gap_all = F.normalize(gap_enc.encode(data.x.to(device)).cpu(), p=2, dim=1)

# ------------------------------------------------------------------
# Step 2: Private Multi-hop Aggregation
# ------------------------------------------------------------------
print('\n--- Step 2: Private Multi-hop Aggregation ---')
set_seed(CONFIG['seed'])
train_hops = private_multi_hop_agg(
    priv_edge_index, x_gap_all, train_nodes,
    sigma=sigma_gap, num_nodes=data.num_nodes, K=GAP_K, seed=CONFIG['seed'])
x_gap_train = torch.cat([x_gap_all[train_nodes]] + train_hops, dim=1)
print(f'  train feature dim: {x_gap_train.size(1)}  ({GAP_K+1} hops × {GAP_ENC_DIM}d)')

# Val: inductive inference — apply PMA to val subgraph with separate noise
val_hops = private_multi_hop_agg(
    val_edge_index, x_gap_all, val_nodes,
    sigma=sigma_gap, num_nodes=data.num_nodes, K=GAP_K, seed=CONFIG['seed'] + 1)
x_gap_val = torch.cat([x_gap_all[val_nodes]] + val_hops, dim=1)

# ------------------------------------------------------------------
# Step 3: Train MLP classifier on concatenated hop features
# ------------------------------------------------------------------
print('\n--- Step 3: GAP classifier (MLP on concatenated hops) ---')
gap_cls_best = train_node_mlp(
    x_gap_train, labels[train_nodes],
    x_gap_val,   labels[val_nodes],
    in_dim=x_gap_train.size(1), hidden=64, num_classes=num_classes,
    epochs=300, lr=0.01, seed=CONFIG['seed'],
)
print(f'\nGAP-EDP best val acc: {gap_cls_best:.4f}')

GAP-EDP  ε=1.0  δ=1e-05  K=1  σ=4.9006

--- Step 1: GAP encoder (MLP, no edges) ---
  encoder train acc (ep 200): 1.0000

--- Step 2: Private Multi-hop Aggregation ---
  train feature dim: 64  (2 hops × 32d)

--- Step 3: GAP classifier (MLP on concatenated hops) ---
  epoch 100 | val_acc=0.7384
  epoch 200 | val_acc=0.7403
  epoch 300 | val_acc=0.7403

GAP-EDP best val acc: 0.7403


In [13]:
# ===================================================================
# CELL 11: Final comparison table
# ===================================================================
print('=' * 58)
print(f'  Cora  |  ε = {CONFIG["epsilon"]}  |  δ = {DELTA}  |  enc-dim = {x_enc.size(1)}')
print('=' * 58)
rows = [
    ('Majority class (no model)',             majority_acc,  'n/a'),
    (f'Gaussian SGC  (σ={sigma_gauss:.2f})', sgc_best_val,  '(ε,δ)-DP'),
    (f'Edge RR       (p_flip={1-math.exp(CONFIG["epsilon"])/(math.exp(CONFIG["epsilon"])+1):.3f})',
                                              rr_best_val,   'ε-DP'),
    (f'GAP-EDP  K={GAP_K}  (σ={sigma_gap:.2f})',
                                              gap_cls_best,  '(ε,δ)-DP'),
    ('MM-EdgeDP / Dict Retrieval',            best_val,      'ε-DP (exp mech)'),
]
for name, acc, dp in rows:
    print(f'  {name:<40}  {acc:.4f}   {dp}')
print('=' * 58)
print('Note: Gaussian SGC and GAP-EDP use (ε,δ)-DP (weaker guarantee).')
print('Edge RR and MM-EdgeDP use pure ε-DP.')

  Cora  |  ε = 1.0  |  δ = 1e-05  |  enc-dim = 32
  Majority class (no model)                 0.3024   n/a
  Gaussian SGC  (σ=4.84)                    0.2746   (ε,δ)-DP
  Edge RR       (p_flip=0.269)              0.3024   ε-DP
  GAP-EDP  K=1  (σ=4.90)                    0.7403   (ε,δ)-DP
  MM-EdgeDP / Dict Retrieval                0.6327   ε-DP (exp mech)
Note: Gaussian SGC and GAP-EDP use (ε,δ)-DP (weaker guarantee).
Edge RR and MM-EdgeDP use pure ε-DP.


In [14]:
# ===================================================================
# CELL 12: Privacy Attack Framework — model storage + helpers
# Threat Model B (white-box): penultimate-layer embeddings, no edges
# Threat Model A (black-box): softmax output vectors, no edges
# Attacks private training edges via logistic regression + AUC-ROC.
# Section 6 of project documentation.
# ===================================================================
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import numpy as np

# ---------------------------------------------------------------
# Retrain all 4 models with same seeds and store references.
# (Training fns in Cells 7-10 kept models in local scope only.)
# ---------------------------------------------------------------
print('Retraining all models for attack evaluation...')

# Gaussian SGC: NodeMLP on noisy 1-hop mean features
set_seed(CONFIG['seed'])
sgc_model = NodeMLP(x_enc.size(1), 64, num_classes).to(device)
_opt = torch.optim.Adam(sgc_model.parameters(), lr=0.01, weight_decay=5e-4)
_xtr, _ytr = x_sgc_train.to(device), labels[train_nodes].to(device)
for _ in range(300):
    sgc_model.train(); _opt.zero_grad()
    F.cross_entropy(sgc_model(_xtr), _ytr).backward(); _opt.step()
sgc_model.eval()
print(f'  SGC stored          (val_acc={sgc_best_val:.4f})')

# Edge-RR: NodeGCN on perturbed graph
set_seed(CONFIG['seed'])
rr_model = NodeGCN(x_enc.size(1), 64, num_classes).to(device)
_opt = torch.optim.Adam(rr_model.parameters(), lr=0.01, weight_decay=5e-4)
_tr_ei_rr, _ = subgraph(train_nodes, perturbed_ei, relabel_nodes=True, num_nodes=data.num_nodes)
_tr_ei_rr = coalesce(to_undirected(_tr_ei_rr, num_nodes=train_nodes.numel()), num_nodes=train_nodes.numel())
_xtr_rr, _ytr_rr = x_enc[train_nodes].to(device), labels[train_nodes].to(device)
_tr_d = _tr_ei_rr.to(device)
for _ in range(300):
    rr_model.train(); _opt.zero_grad()
    F.cross_entropy(rr_model(_xtr_rr, _tr_d), _ytr_rr).backward(); _opt.step()
rr_model.eval()
print(f'  Edge-RR stored      (val_acc={rr_best_val:.4f})')

# GAP-EDP: gap_enc is global from Cell 10; retrain NodeMLP classifier
set_seed(CONFIG['seed'])
gap_cls_model = NodeMLP(x_gap_train.size(1), 64, num_classes).to(device)
_opt = torch.optim.Adam(gap_cls_model.parameters(), lr=0.01, weight_decay=5e-4)
_xtr_gap, _ytr_gap = x_gap_train.to(device), labels[train_nodes].to(device)
for _ in range(300):
    gap_cls_model.train(); _opt.zero_grad()
    F.cross_entropy(gap_cls_model(_xtr_gap), _ytr_gap).backward(); _opt.step()
gap_cls_model.eval()
print(f'  GAP-EDP stored      (val_acc={gap_cls_best:.4f})')

# MM-EdgeDP: StandardGCN on prototype assignments
set_seed(CONFIG['seed'])
mm_model = StandardGCN(32, x_enc.size(1), num_classes).to(device)
_opt = torch.optim.Adam(mm_model.parameters(), lr=CONFIG['lr'])
_mm_crit = nn.CrossEntropyLoss()
_mm_ds = PrototypeAssignmentDataset(public_dict, assignments['proto_indices'], assignments['labels'])
_mm_dl = DataLoader(_mm_ds, batch_size=CONFIG['batch_size'], shuffle=True)
for _ in range(CONFIG['epochs']):
    mm_model.train()
    for _b in _mm_dl:
        _b = _b.to(device); _opt.zero_grad(set_to_none=True)
        _xa = _b.x + torch.randn_like(_b.x) * CONFIG['feature_jitter_std']
        _ea, _ = dropout_edge(_b.edge_index, p=CONFIG['edge_dropout_p'])
        if _ea.numel() == 0: _ea = _b.edge_index
        _mm_crit(mm_model(_xa, _ea, _b.batch), _b.y).backward(); _opt.step()
mm_model.eval()
print(f'  MM-EdgeDP stored    (val_acc={best_val:.4f})')

# ---------------------------------------------------------------
# White-box (Threat Model B): penultimate-layer embeddings.
# Queried with node's own feature vector, no edges (isolated).
# With empty edge_index, GCNConv reduces to a linear transform.
# ---------------------------------------------------------------
_EMPTY_EI = torch.zeros(2, 0, dtype=torch.long, device=device)


@torch.no_grad()
def wb_embed_sgc(model, x_features, node_ids):
    """NodeMLP: Linear + ReLU → [N, 64]."""
    x = x_features[node_ids].to(device)
    return model.net[1](model.net[0](x)).cpu()          # net[0]=Linear, net[1]=ReLU


@torch.no_grad()
def wb_embed_rr(model, x_features, node_ids):
    """NodeGCN: conv1 + ReLU, empty edges → [N, 64]."""
    x = x_features[node_ids].to(device)
    return model.conv1(x, _EMPTY_EI).relu().cpu()


@torch.no_grad()
def wb_embed_gap(enc_model, x_raw, node_ids):
    """GAPEncoder.encode() + L2-normalize → [N, 32]."""
    h = enc_model.encode(x_raw[node_ids].to(device))
    return F.normalize(h, p=2, dim=1).cpu()


@torch.no_grad()
def wb_embed_mm(model, x_features, node_ids):
    """StandardGCN: conv1 + conv2 + ReLU, empty edges → [N, 32]."""
    x = x_features[node_ids].to(device)
    h = model.conv1(x, _EMPTY_EI).relu()
    return model.conv2(h, _EMPTY_EI).relu().cpu()


# ---------------------------------------------------------------
# Black-box (Threat Model A): softmax output vectors.
# Queried with node's own feature vector, no edges (isolated).
# ---------------------------------------------------------------

@torch.no_grad()
def bb_probs_sgc(model, x_features, node_ids):
    """NodeMLP softmax → [N, C]."""
    x = x_features[node_ids].to(device)
    return torch.softmax(model(x), dim=1).cpu()


@torch.no_grad()
def bb_probs_rr(model, x_features, node_ids):
    """NodeGCN softmax, empty edges → [N, C]."""
    x = x_features[node_ids].to(device)
    return torch.softmax(model(x, _EMPTY_EI), dim=1).cpu()


@torch.no_grad()
def bb_probs_gap(enc_model, cls_model, x_raw, node_ids):
    """GAP isolated query: encode → concat zeros (no-neighbor hop) → cls softmax → [N, C]."""
    h_enc = F.normalize(enc_model.encode(x_raw[node_ids].to(device)), p=2, dim=1)
    h_cat = torch.cat([h_enc, torch.zeros_like(h_enc)], dim=1)   # zeros = no-hop signal
    return torch.softmax(cls_model(h_cat), dim=1).cpu()


@torch.no_grad()
def bb_probs_mm(model, x_features, node_ids):
    """StandardGCN: isolated forward (empty edges) → pool → softmax → [N, C]."""
    x = x_features[node_ids].to(device)
    h = model.conv1(x, _EMPTY_EI).relu()
    h = model.conv2(h, _EMPTY_EI).relu()                              # [N, 32]
    batch = torch.arange(node_ids.numel(), device=device)             # each node = own graph
    h_pool = global_mean_pool(h, batch)                               # [N, 32]
    return torch.softmax(model.lin(h_pool), dim=1).cpu()


# ---------------------------------------------------------------
# Pair construction, pairwise feature builders, and LR attack
# ---------------------------------------------------------------

def build_attack_pairs(edge_index, node_ids, n_max, seed):
    """
    Returns (pos_pairs, neg_pairs) as [M, 2] local-index tensors.
    pos_pairs: unique undirected real private edges (capped at n_max).
    neg_pairs: equal number of randomly sampled non-edges.
    """
    rng = np.random.default_rng(seed)
    node_list = node_ids.tolist()
    local = {v: i for i, v in enumerate(node_list)}
    n = len(node_list)

    pos_set = set()
    for u, v in zip(edge_index[0].tolist(), edge_index[1].tolist()):
        if u in local and v in local and u != v:
            a, b = local[u], local[v]
            pos_set.add((min(a, b), max(a, b)))
    pos_list = list(pos_set)
    rng.shuffle(pos_list)
    pos_list = pos_list[:n_max]

    neg_list, all_edges = [], set(pos_set)
    while len(neg_list) < len(pos_list):
        i = int(rng.integers(0, n)); j = int(rng.integers(0, n))
        if i != j:
            key = (min(i, j), max(i, j))
            if key not in all_edges:
                neg_list.append(key); all_edges.add(key)

    pos = torch.tensor(pos_list, dtype=torch.long)
    neg = torch.tensor(neg_list, dtype=torch.long)
    print(f'  {len(pos_list)} positive + {len(neg_list)} negative pairs '
          f'({n} train nodes, {n * (n - 1) // 2:,} total candidate pairs)')
    return pos, neg


def pairwise_wb(embeddings, pairs):
    """Hadamard product h_u ⊙ h_v → [M, d] (standard link-pred construction)."""
    return (embeddings[pairs[:, 0]] * embeddings[pairs[:, 1]]).numpy()


def pairwise_bb(probs, pairs):
    """[|p_u − p_v|, p_u ⊙ p_v] → [M, 2·C] (captures disagreement + joint activation)."""
    pu, pv = probs[pairs[:, 0]], probs[pairs[:, 1]]
    return torch.cat([(pu - pv).abs(), pu * pv], dim=1).numpy()


def run_attack(feats_pos, feats_neg, test_frac=0.30, seed=0):
    """Train logistic regression on pairwise features; return AUC-ROC on held-out test."""
    X = np.vstack([feats_pos, feats_neg])
    y = np.array([1] * len(feats_pos) + [0] * len(feats_neg), dtype=int)
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(y))
    X, y = X[idx], y[idx]
    n_te = max(2, int(len(y) * test_frac))
    clf = LogisticRegression(max_iter=2000, C=1.0, solver='lbfgs')
    clf.fit(X[n_te:], y[n_te:])
    return roc_auc_score(y[:n_te], clf.predict_proba(X[:n_te])[:, 1])


print('\nAttack helpers ready.')

Retraining all models for attack evaluation...
  SGC stored          (val_acc=0.2746)
  Edge-RR stored      (val_acc=0.3024)
  GAP-EDP stored      (val_acc=0.7403)
  MM-EdgeDP stored    (val_acc=0.6327)

Attack helpers ready.


In [15]:
# ===================================================================
# CELL 13: Run privacy attacks and comparison table
# ===================================================================
N_ATTACK = 1721   # target pairs per class; capped by actual edge count
ATK_SEED  = CONFIG['seed']

print('Building attack dataset from private training edges...')
pos_pairs, neg_pairs = build_attack_pairs(
    priv_edge_index, train_nodes, n_max=N_ATTACK, seed=ATK_SEED)

attack_results = {}   # {model_name: {'wb': auc, 'bb': auc}}


def eval_attack_model(name, wb_fn, bb_fn):
    print(f'\n[{name}]')
    h = wb_fn()
    auc_wb = run_attack(pairwise_wb(h, pos_pairs), pairwise_wb(h, neg_pairs), seed=ATK_SEED)
    print(f'  Threat Model B (white-box)  AUC = {auc_wb:.4f}')
    p = bb_fn()
    auc_bb = run_attack(pairwise_bb(p, pos_pairs), pairwise_bb(p, neg_pairs), seed=ATK_SEED)
    print(f'  Threat Model A (black-box)  AUC = {auc_bb:.4f}')
    attack_results[name] = {'wb': auc_wb, 'bb': auc_bb}


eval_attack_model(
    'Gaussian SGC',
    lambda: wb_embed_sgc(sgc_model, x_enc, train_nodes),
    lambda: bb_probs_sgc(sgc_model, x_enc, train_nodes),
)
eval_attack_model(
    'Edge RR',
    lambda: wb_embed_rr(rr_model, x_enc, train_nodes),
    lambda: bb_probs_rr(rr_model, x_enc, train_nodes),
)
eval_attack_model(
    'GAP-EDP',
    lambda: wb_embed_gap(gap_enc, data.x, train_nodes),
    lambda: bb_probs_gap(gap_enc, gap_cls_model, data.x, train_nodes),
)
eval_attack_model(
    'MM-EdgeDP',
    lambda: wb_embed_mm(mm_model, x_enc, train_nodes),
    lambda: bb_probs_mm(mm_model, x_enc, train_nodes),
)

# ---------------------------------------------------------------
# Comparison table
# ---------------------------------------------------------------
_val_accs = {
    'Gaussian SGC': sgc_best_val,
    'Edge RR':      rr_best_val,
    'GAP-EDP':      gap_cls_best,
    'MM-EdgeDP':    best_val,
}
_dp_type = {
    'Gaussian SGC': '(ε,δ)-DP',
    'Edge RR':      'ε-DP',
    'GAP-EDP':      '(ε,δ)-DP',
    'MM-EdgeDP':    'ε-DP (exp mech)',
}

print('\n' + '=' * 73)
print(f'  Edge Inference Attack AUC  |  Cora  |  ε = {CONFIG["epsilon"]}  |  δ = {DELTA}')
print('=' * 73)
print(f'  {"Model":<20}  {"DP Type":<16}  {"Val Acc":>8}  '
      f'{"WB AUC (B)":>10}  {"BB AUC (A)":>10}')
print('-' * 73)
for name, res in attack_results.items():
    print(f'  {name:<20}  {_dp_type[name]:<16}  {_val_accs[name]:>8.4f}  '
          f'{res["wb"]:>10.4f}  {res["bb"]:>10.4f}')
print('=' * 73)
print('AUC ≈ 0.5 → no leakage (random guessing)  |  AUC ≈ 1.0 → full leakage')
print('WB = white-box adversary (weights + embeddings)  |  BB = black-box (API only)')

Building attack dataset from private training edges...
  1721 positive + 1721 negative pairs (1630 train nodes, 1,327,635 total candidate pairs)

[Gaussian SGC]
  Threat Model B (white-box)  AUC = 0.5000
  Threat Model A (black-box)  AUC = 0.5000

[Edge RR]
  Threat Model B (white-box)  AUC = 0.5360
  Threat Model A (black-box)  AUC = 0.6246

[GAP-EDP]
  Threat Model B (white-box)  AUC = 0.8811
  Threat Model A (black-box)  AUC = 0.8551

[MM-EdgeDP]
  Threat Model B (white-box)  AUC = 0.7267
  Threat Model A (black-box)  AUC = 0.6978

  Edge Inference Attack AUC  |  Cora  |  ε = 1.0  |  δ = 1e-05
  Model                 DP Type            Val Acc  WB AUC (B)  BB AUC (A)
-------------------------------------------------------------------------
  Gaussian SGC          (ε,δ)-DP            0.2746      0.5000      0.5000
  Edge RR               ε-DP                0.3024      0.5360      0.6246
  GAP-EDP               (ε,δ)-DP            0.7403      0.8811      0.8551
  MM-EdgeDP           